In [ ]:
"""
================================================================================
  Microbiota Similarity Score (MSS) — Machine Learning Workflow
  Gut Microbiota Analysis in Alzheimer's Disease & JSHT Intervention

  Study:  A Machine Learning Framework for Assessing Gut Microbiota Similarity
          in Alzheimer's Disease: A Proof-of-Concept Application to a
          Jing Si Herbal Tea Intervention Cohort

  Code written on this 2nd day of September 2025

================================================================================
"""

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  We begin with IMPORTS as always
# ──────────────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    f1_score, brier_score_loss
)
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, cross_val_predict
)
from sklearn.pipeline import Pipeline

import shap

# Reproducibility seeds used across the entire workflow
RANDOM_SEEDS = [42, 123, 456]
OUTER_FOLDS  = 10
INNER_FOLDS  = 5

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 1 — LOAD & PREPROCESS EXTERNAL DATASET
# ──────────────────────────────────────────────────────────────────────────────

external_df = pd.read_excel("External.xlsx")

# Encode target: NC = 1 (positive class — cognitively normal)
#                AD = 0
label_encoder = LabelEncoder()
# Fit so that NC maps to 1 (alphabetically NC > AD, so AD=0, NC=1)
label_encoder.fit(["AD", "NC"])
y_external = label_encoder.transform(external_df["Status"])   # AD=0, NC=1

# Feature matrix: drop sample ID (Genus) and Status
feature_cols_external = [
    c for c in external_df.columns if c not in ["Genus", "Status"]
]
X_external = external_df[feature_cols_external].values
genus_names = feature_cols_external   # preserve genus names for SHAP

print(f"    External dataset: {X_external.shape[0]} samples, "
      f"{X_external.shape[1]} genera")
print(f"    AD = {(y_external == 0).sum()}, NC = {(y_external == 1).sum()}")

# CLR transformation (centered log-ratio)
# Pseudocount 1e-6 added to handle zero inflation
def clr_transform(X, pseudocount=1e-6):
    X_pseudo = X + pseudocount
    log_X    = np.log(X_pseudo)
    clr      = log_X - log_X.mean(axis=1, keepdims=True)
    return clr

X_external_clr = clr_transform(X_external)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 2 — NESTED CROSS-VALIDATION: RF, SVM-RBF, XGBOOST, L1-LR
# ──────────────────────────────────────────────────────────────────────────────

# Hyperparameter grids (literature-grounded ranges for microbiome ML)
# Reference: Topçuoğlu et al. 2020 mBio; Pasolli et al. 2019 Cell Host Microbe
param_grids = {
    "Random Forest": {
        "clf__n_estimators":      [100, 200, 300, 500],
        "clf__max_depth":         [None, 5, 10, 20],
        "clf__min_samples_leaf":  [1, 2, 4],
        "clf__max_features":      ["sqrt", "log2"],
    },
    "SVM-RBF": {
        "clf__C":     [0.01, 0.1, 1, 10, 100],
        "clf__gamma": [0.001, 0.01, 0.1, "scale"],
    },
    "XGBoost": {
        "clf__n_estimators":       [100, 200, 300, 500],
        "clf__max_depth":          [3, 5, 7, 9],
        "clf__learning_rate":      [0.01, 0.05, 0.1, 0.3],
        "clf__subsample":          [0.6, 0.8, 1.0],
        "clf__colsample_bytree":   [0.6, 0.8, 1.0],
    },
    "L1-Logistic Regression": {
        "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    },
}

# Base estimators (class_weight='balanced' addresses AD/NC imbalance)
base_estimators = {
    "Random Forest": RandomForestClassifier(
        class_weight="balanced", random_state=42
    ),
    "SVM-RBF": SVC(
        kernel="rbf", probability=True,
        class_weight="balanced", random_state=42
    ),
    "XGBoost": XGBClassifier(
        eval_metric="logloss", random_state=42,
        use_label_encoder=False,
        scale_pos_weight=(y_external == 0).sum() / (y_external == 1).sum()
    ),
    "L1-Logistic Regression": LogisticRegression(
        penalty="l1", solver="liblinear",
        class_weight="balanced", max_iter=1000, random_state=42
    ),
}

def nested_cv_evaluate(estimator, param_grid, X, y,
                        outer_folds=OUTER_FOLDS,
                        inner_folds=INNER_FOLDS,
                        seeds=RANDOM_SEEDS):
    """
    Stratified nested cross-validation repeated across multiple seeds.
    Returns dict of mean ± std for AUC, accuracy, macro precision, macro F1.
    Also returns all outer-fold predicted probabilities for SHAP aggregation.
    """
    all_aucs, all_accs, all_precs, all_f1s = [], [], [], []
    all_proba = np.zeros(len(y))   # aggregated probability (NC class)

    for seed in seeds:
        outer_cv = StratifiedKFold(
            n_splits=outer_folds, shuffle=True, random_state=seed
        )
        fold_proba = np.zeros(len(y))

        for train_idx, test_idx in outer_cv.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            # Inner CV for hyperparameter search
            inner_cv = StratifiedKFold(
                n_splits=inner_folds, shuffle=True, random_state=seed
            )
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    estimator)
            ])
            grid = GridSearchCV(
                pipe, param_grid, cv=inner_cv,
                scoring="roc_auc", n_jobs=-1, refit=True
            )
            grid.fit(X_train, y_train)
            best = grid.best_estimator_

            # Evaluate on outer fold
            proba_test = best.predict_proba(X_test)[:, 1]   # P(NC)
            y_pred     = best.predict(X_test)

            fold_proba[test_idx] = proba_test
            all_aucs.append(roc_auc_score(y_test, proba_test))
            all_accs.append(accuracy_score(y_test, y_pred))
            all_precs.append(precision_score(
                y_test, y_pred, average="macro", zero_division=0
            ))
            all_f1s.append(f1_score(
                y_test, y_pred, average="macro", zero_division=0
            ))

        all_proba += fold_proba

    all_proba /= len(seeds)   # average across seeds

    return {
        "AUC_mean":  np.mean(all_aucs),
        "AUC_std":   np.std(all_aucs),
        "AUC_95CI":  (
            np.mean(all_aucs) - 1.96 * np.std(all_aucs) / np.sqrt(len(all_aucs)),
            np.mean(all_aucs) + 1.96 * np.std(all_aucs) / np.sqrt(len(all_aucs))
        ),
        "Acc_mean":  np.mean(all_accs),
        "Acc_std":   np.std(all_accs),
        "Prec_mean": np.mean(all_precs),
        "Prec_std":  np.std(all_precs),
        "F1_mean":   np.mean(all_f1s),
        "F1_std":    np.std(all_f1s),
        "cv_proba":  all_proba,
    }

performance_results = {}
cv_proba_store      = {}

for model_name, estimator in base_estimators.items():
    print(f"    Running nested CV: {model_name} ...")
    results = nested_cv_evaluate(
        estimator        = estimator,
        param_grid       = param_grids[model_name],
        X                = X_external_clr,
        y                = y_external,
    )
    performance_results[model_name] = results
    cv_proba_store[model_name]      = results["cv_proba"]
    print(f"      AUC = {results['AUC_mean']:.3f} ± {results['AUC_std']:.3f}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 3 — SELECT THE BEST PERFORMED MODEL AS FINAL MODEL;CALIBRATION (BRIER SCORE)
# ──────────────────────────────────────────────────────────────────────────────

# SVM-RBF was selected as the final model per study design
# Final model was trained on ALL external data for subsequent MSS derivation

scaler_final = StandardScaler()
X_external_scaled = scaler_final.fit_transform(X_external_clr)

# Best hyperparameters informed by nested CV literature defaults for microbiome
svm_base = SVC(
    kernel="rbf", C=1.0, gamma="scale",
    probability=True, class_weight="balanced", random_state=42
)

# Platt scaling calibration via isotonic regression — 5-fold CV
svm_calibrated = CalibratedClassifierCV(
    svm_base, cv=5, method="sigmoid"   # sigmoid = Platt scaling
)
svm_calibrated.fit(X_external_scaled, y_external)

# Brier score on cross-validated predictions (honest estimate)
cv_proba_svm = cv_proba_store["SVM-RBF"]
brier = brier_score_loss(y_external, cv_proba_svm)
print(f"    SVM-RBF Brier Score (cross-validated) = {brier:.4f}")

# Calibration curve
fraction_pos, mean_pred = calibration_curve(
    y_external, cv_proba_svm, n_bins=10, strategy="uniform"
)

fig_cal, ax_cal = plt.subplots(figsize=(6, 5))
ax_cal.plot(mean_pred, fraction_pos, "s-", color="#1D9E75",
            label=f"SVM-RBF (Brier = {brier:.3f})", linewidth=2)
ax_cal.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
ax_cal.set_xlabel("Mean predicted probability (P(NC))", fontsize=12)
ax_cal.set_ylabel("Fraction of positives (NC)", fontsize=12)
ax_cal.set_title("SVM-RBF Probability Calibration Curve\n"
                 "(Cross-validated on External dataset)", fontsize=12)
ax_cal.legend(fontsize=10)
ax_cal.set_xlim([0, 1])
ax_cal.set_ylim([0, 1])
sns.despine()
fig_cal.tight_layout()
fig_cal.savefig("Figure_Calibration_Curve.png", dpi=300, bbox_inches="tight")
plt.close(fig_cal)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 4 — DERIVE MSS ON EXTERNAL DATASET
# ──────────────────────────────────────────────────────────────────────────────

# MSS = P(NC) from calibrated SVM-RBF
# Higher MSS → more similar to cognitively normal microbiota profile
mss_external = svm_calibrated.predict_proba(X_external_scaled)[:, 1]

external_df["MSS"] = mss_external
external_df["MSS_group"] = external_df["Status"]   # AD or NC

mss_ad = mss_external[y_external == 0]
mss_nc = mss_external[y_external == 1]

print(f"    MSS in AD samples: mean = {mss_ad.mean():.3f}, "
      f"SD = {mss_ad.std():.3f}")
print(f"    MSS in NC samples: mean = {mss_nc.mean():.3f}, "
      f"SD = {mss_nc.std():.3f}")

# Performance summary table
perf_table = []
for model_name, res in performance_results.items():
    perf_table.append({
        "Model":              model_name,
        "AUC (mean ± SD)":   f"{res['AUC_mean']:.3f} ± {res['AUC_std']:.3f}",
        "95% CI":             f"({res['AUC_95CI'][0]:.3f}, {res['AUC_95CI'][1]:.3f})",
        "Accuracy (mean ± SD)":  f"{res['Acc_mean']:.3f} ± {res['Acc_std']:.3f}",
        "Macro Precision":    f"{res['Prec_mean']:.3f} ± {res['Prec_std']:.3f}",
        "Macro F1-Score":     f"{res['F1_mean']:.3f} ± {res['F1_std']:.3f}",
    })

perf_df = pd.DataFrame(perf_table)
perf_df.to_excel("Table_Model_Performance.xlsx", index=False)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 5 — LOAD & PREPROCESS JSHT DATASET; APPLY MSS
# ──────────────────────────────────────────────────────────────────────────────

jsht_df = pd.read_excel("JSHT.xlsx")

# Strip leading whitespace from JSHT genus column names
jsht_df.columns = jsht_df.columns.str.strip()

# Confirm genus columns align with external feature columns
missing_genera = [g for g in feature_cols_external if g not in jsht_df.columns]
if missing_genera:
    print(f"    WARNING: {len(missing_genera)} genera missing from JSHT — "
          f"filling with 0")
    for g in missing_genera:
        jsht_df[g] = 0.0

# Extract JSHT feature matrix using the SAME genus order as external training
X_jsht_raw = jsht_df[feature_cols_external].values

# CLR transform using EXTERNAL scaling parameters (no data leakage)
# We apply CLR to JSHT and then scale using the external scaler
X_jsht_clr    = clr_transform(X_jsht_raw)
X_jsht_scaled = scaler_final.transform(X_jsht_clr)   # external scaler

# Generate MSS for each JSHT sample
mss_jsht = svm_calibrated.predict_proba(X_jsht_scaled)[:, 1]
jsht_df["MSS"] = mss_jsht

print(f"    JSHT dataset: {jsht_df.shape[0]} samples across "
      f"{jsht_df['patient'].nunique()} groups and "
      f"{jsht_df['week'].nunique()} timepoints.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 6 — MSS SIMILARITY TABLE PER SAMPLE PER TIMEPOINT
# ──────────────────────────────────────────────────────────────────────────────

# Assign sample-level IDs within each group × timepoint
jsht_df["Sample_ID"] = (
    jsht_df["patient"] + "_" +
    jsht_df.groupby(["patient", "week"]).cumcount().add(1).astype(str)
)

mss_table = jsht_df[["patient", "week", "Sample_ID", "MSS"]].copy()
mss_table.columns = ["Group", "Timepoint", "Sample_ID", "MSS (P(NC))"]
mss_table["MSS (P(NC))"] = mss_table["MSS (P(NC))"].round(4)
mss_table["Timepoint"] = pd.Categorical(
    mss_table["Timepoint"], categories=["WK0", "WK12", "WK24"], ordered=True
)
mss_table = mss_table.sort_values(["Group", "Timepoint"]).reset_index(drop=True)

mss_table.to_excel("Table_MSS_Similarity_Scores.xlsx", index=False)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 7 — LONGITUDINAL MSS LINE PLOT — TA vs TB
# ──────────────────────────────────────────────────────────────────────────────

week_order = ["WK0", "WK12", "WK24"]
week_labels = ["Baseline\n(WK0)", "Week 12\n(WK12)", "Week 24\n(WK24)"]

# Compute mean MSS per group × timepoint
mss_summary = (
    jsht_df.groupby(["patient", "week"])["MSS"]
    .agg(Mean="mean", SD="std", SE=lambda x: x.std() / np.sqrt(len(x)))
    .reset_index()
)
mss_summary["week"] = pd.Categorical(
    mss_summary["week"], categories=week_order, ordered=True
)
mss_summary = mss_summary.sort_values(["patient", "week"])

ta_data = mss_summary[mss_summary["patient"] == "TA"].set_index("week")
tb_data = mss_summary[mss_summary["patient"] == "TB"].set_index("week")

fig_long, ax_long = plt.subplots(figsize=(7, 5))

x_pos = np.arange(len(week_order))

# TA line
ax_long.errorbar(
    x_pos, ta_data.loc[week_order, "Mean"],
    yerr=ta_data.loc[week_order, "SE"],
    fmt="o-", color="#1D9E75", linewidth=2.2, markersize=8,
    capsize=5, label="Group TA (3g daily)"
)

# TB line
ax_long.errorbar(
    x_pos, tb_data.loc[week_order, "Mean"],
    yerr=tb_data.loc[week_order, "SE"],
    fmt="s--", color="#D85A30", linewidth=2.2, markersize=8,
    capsize=5, label="Group TB (6g daily)"
)

# Individual sample trajectories (light, for transparency)
for group, color in [("TA", "#1D9E75"), ("TB", "#D85A30")]:
    group_data = jsht_df[jsht_df["patient"] == group]
    for sample_id in group_data["Sample_ID"].unique():
        samp = group_data[group_data["Sample_ID"] == sample_id].set_index("week")
        try:
            y_vals = [samp.loc[w, "MSS"] for w in week_order if w in samp.index]
            x_vals = [i for i, w in enumerate(week_order) if w in samp.index]
            ax_long.plot(x_vals, y_vals, color=color,
                         alpha=0.18, linewidth=1.0)
        except Exception:
            pass

ax_long.set_xticks(x_pos)
ax_long.set_xticklabels(week_labels, fontsize=11)
ax_long.set_ylabel("Microbiota Similarity Score (MSS)\nP(similar to NC profile)",
                   fontsize=11)
ax_long.set_xlabel("Intervention Timepoint", fontsize=11)
ax_long.set_title(
    "Longitudinal MSS Trajectories: JSHT Intervention Cohort\n"
    "Group TA (3g/day) vs Group TB (6g/day) — 24-Week Follow-up",
    fontsize=11
)
ax_long.set_ylim([0, 1])
ax_long.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax_long.legend(fontsize=10, framealpha=0.8)
ax_long.axhline(
    y=mss_nc.mean(), color="#444441", linestyle=":", linewidth=1.4,
    label="NC reference mean"
)
ax_long.text(
    2.05, mss_nc.mean() + 0.01,
    f"NC ref. (mean={mss_nc.mean():.2f})",
    fontsize=9, color="#444441"
)
ax_long.annotate(
    "Exploratory · not FDR-corrected · n=8 (TA), n=7 (TB)",
    xy=(0.01, 0.01), xycoords="axes fraction",
    fontsize=8, color="gray", style="italic"
)
sns.despine()
fig_long.tight_layout()
fig_long.savefig("Figure_Longitudinal_MSS.png", dpi=300, bbox_inches="tight")
plt.close(fig_long)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 8 — SHAP ON EXTERNAL DATASET — TOP 10 GENERA
# ──────────────────────────────────────────────────────────────────────────────
# SHAP requires a predict function; we use calibrated SVM probability
# Background dataset: random sample of 100 from external data
np.random.seed(42)
background_idx = np.random.choice(
    X_external_scaled.shape[0], size=100, replace=False
)
X_background = X_external_scaled[background_idx]

# KernelExplainer — model-agnostic, appropriate for SVM
explainer = shap.KernelExplainer(
    model   = svm_calibrated.predict_proba,
    data    = X_background,
    link    = "identity"
)

# Compute SHAP values on the held-out outer fold test samples
# (aggregated cv proba gives us an honest ranking)
# For SHAP we use a representative subset to keep runtime manageable
shap_subset_idx = np.random.choice(
    X_external_scaled.shape[0], size=min(150, X_external_scaled.shape[0]),
    replace=False
)
X_shap_subset = X_external_scaled[shap_subset_idx]
y_shap_subset = y_external[shap_subset_idx]

print("    Computing SHAP values (this may take a few minutes) ...")
shap_values_all = explainer.shap_values(X_shap_subset)

# shap_values_all[1] = SHAP values for class NC (positive class)
shap_nc = shap_values_all[1]   # shape: (n_samples, n_features)

# Mean absolute SHAP per genus → ranking
mean_abs_shap = np.abs(shap_nc).mean(axis=0)
shap_series   = pd.Series(mean_abs_shap, index=genus_names)
top10_genera  = shap_series.nlargest(10).index.tolist()

print(f"    Top 10 genera by mean |SHAP|: {top10_genera}")

# Classify direction: mean SHAP > 0 → NC-associated; < 0 → AD-associated
mean_shap_direction = shap_nc.mean(axis=0)
nc_genera  = [g for g in top10_genera if mean_shap_direction[genus_names.index(g)] < 0]
ad_genera  = [g for g in top10_genera if mean_shap_direction[genus_names.index(g)] > 0]
# Note: negative SHAP for NC class = contributes to AD prediction;
#       positive SHAP for NC class = contributes toward NC classification
# Redefine: positive SHAP = NC-elevated, negative SHAP = AD-elevated
nc_genera  = [g for g in top10_genera if mean_shap_direction[genus_names.index(g)] > 0]
ad_genera  = [g for g in top10_genera if mean_shap_direction[genus_names.index(g)] <= 0]

print(f"    NC-elevated genera (top 3 used for boxplots): "
      f"{nc_genera[:3]}")
print(f"    AD-elevated genera (top 2 used for boxplots): "
      f"{ad_genera[:2]}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 9 — SHAP BEESWARM PLOT ON JSHT DATASET (TOP 10 GENERA)
# ──────────────────────────────────────────────────────────────────────────────

# Compute SHAP values for all JSHT samples using external-trained explainer
print("    Computing SHAP values for JSHT samples ...")
shap_values_jsht = explainer.shap_values(X_jsht_scaled)
shap_nc_jsht     = shap_values_jsht[1]   # NC class SHAP values

# Restrict to top 10 genera indices
top10_idx    = [genus_names.index(g) for g in top10_genera]
shap_top10   = shap_nc_jsht[:, top10_idx]
X_top10_jsht = X_jsht_scaled[:, top10_idx]

# Beeswarm (summary) plot using SHAP
shap_explanation = shap.Explanation(
    values          = shap_top10,
    base_values     = np.full(shap_top10.shape[0], explainer.expected_value[1]),
    data            = X_top10_jsht,
    feature_names   = top10_genera
)

fig_shap, ax_shap = plt.subplots(figsize=(9, 6))
shap.plots.beeswarm(
    shap_explanation,
    max_display = 10,
    show        = False,
    color_bar   = True
)
plt.title(
    "KernelSHAP Beeswarm Plot — Top 10 Microbial Genera\n"
    "Applied to JSHT Intervention Cohort (n=45)\n"
    "Positive SHAP → NC classification | Negative SHAP → AD classification",
    fontsize=10
)
plt.tight_layout()
plt.savefig("Figure_SHAP_Beeswarm_JSHT.png", dpi=300, bbox_inches="tight")
plt.close()


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
#  PART 10 — BOXPLOTS: NC-ELEVATED & AD-ELEVATED GENERA IN JSHT DATA
# ──────────────────────────────────────────────────────────────────────────────

# NC-elevated (beneficial): Faecalibacterium, Akkermansia, Tyzzerella
# AD-elevated (dysbiosis):  Streptococcus, Lentihominibacter
NC_GENERA_BOXPLOT = ["Faecalibacterium", "Akkermansia", "Tyzzerella"]
AD_GENERA_BOXPLOT = ["Streptococcus", "Lentihominibacter"]

# Prepare long-form JSHT dataframe for boxplots
jsht_long = jsht_df[
    ["patient", "week"] + NC_GENERA_BOXPLOT + AD_GENERA_BOXPLOT
].copy()

jsht_long["week"] = pd.Categorical(
    jsht_long["week"], categories=["WK0", "WK12", "WK24"], ordered=True
)

week_palette = {
    "WK0":  "#B5D4F4",
    "WK12": "#85B7EB",
    "WK24": "#1D9E75",
}

# ── Figure A: NC-elevated genera (3 panels) ──────────────────────────────────
fig_nc, axes_nc = plt.subplots(1, 3, figsize=(14, 5), sharey=False)
fig_nc.suptitle(
    "Relative Abundance of NC-Associated Genera in JSHT Cohort\n"
    "(Faecalibacterium · Akkermansia · Tyzzerella)",
    fontsize=12, fontweight="bold"
)

for ax, genus in zip(axes_nc, NC_GENERA_BOXPLOT):
    for i, group in enumerate(["TA", "TB"]):
        grp_data = jsht_long[jsht_long["patient"] == group]
        sns.boxplot(
            data      = grp_data,
            x         = "week",
            y         = genus,
            palette   = week_palette,
            order      = ["WK0", "WK12", "WK24"],
            ax         = ax,
            width      = 0.55,
            linewidth  = 1.2,
            fliersize  = 3,
            showfliers = True,
            boxprops   = dict(alpha=0.8 if group == "TA" else 0.5),
            medianprops= dict(color="black", linewidth=1.8),
        )

    ax.set_title(genus, fontsize=11, style="italic")
    ax.set_xlabel("Timepoint", fontsize=10)
    ax.set_ylabel("Relative Abundance", fontsize=10)
    ax.set_xticklabels(["Baseline", "Week 12", "Week 24"], fontsize=9)
    ax.annotate(
        "TA (solid) | TB (faded)",
        xy=(0.5, -0.18), xycoords="axes fraction",
        ha="center", fontsize=8, color="gray"
    )
    sns.despine(ax=ax)

fig_nc.tight_layout()
fig_nc.savefig("Figure_Boxplot_NC_Genera.png", dpi=300, bbox_inches="tight")
plt.close(fig_nc)
print("    Saved: Figure_Boxplot_NC_Genera.png")

# ── Figure B: AD-elevated genera (2 panels) ──────────────────────────────────
ad_palette = {
    "WK0":  "#F5C4B3",
    "WK12": "#F0997B",
    "WK24": "#D85A30",
}

fig_ad, axes_ad = plt.subplots(1, 2, figsize=(10, 5), sharey=False)
fig_ad.suptitle(
    "Relative Abundance of AD-Associated Genera in JSHT Cohort\n"
    "(Streptococcus · Lentihominibacter)",
    fontsize=12, fontweight="bold"
)

for ax, genus in zip(axes_ad, AD_GENERA_BOXPLOT):
    for group in ["TA", "TB"]:
        grp_data = jsht_long[jsht_long["patient"] == group]
        sns.boxplot(
            data       = grp_data,
            x          = "week",
            y          = genus,
            palette    = ad_palette,
            order       = ["WK0", "WK12", "WK24"],
            ax          = ax,
            width       = 0.55,
            linewidth   = 1.2,
            fliersize   = 3,
            showfliers  = True,
            boxprops    = dict(alpha=0.85 if group == "TA" else 0.5),
            medianprops = dict(color="black", linewidth=1.8),
        )

    ax.set_title(genus, fontsize=11, style="italic")
    ax.set_xlabel("Timepoint", fontsize=10)
    ax.set_ylabel("Relative Abundance", fontsize=10)
    ax.set_xticklabels(["Baseline", "Week 12", "Week 24"], fontsize=9)
    ax.annotate(
        "TA (solid) | TB (faded)",
        xy=(0.5, -0.18), xycoords="axes fraction",
        ha="center", fontsize=8, color="gray"
    )
    sns.despine(ax=ax)

fig_ad.tight_layout()
fig_ad.savefig("Figure_Boxplot_AD_Genera.png", dpi=300, bbox_inches="tight")
plt.close(fig_ad)


In [1]:
print("And with that my friend, i hope this will be all well")

And with that my friend, i hope this will be all well
